# ЛР2. Цифровой двойник, этап 2: углы коммутации и предельная характеристика
Предварительно найдите режим в интерактивном тренажере (ссылка в РПД, п. 3.2).

In [ ]:
# Подготовка среды: папки scripts, autograder и detective должны лежать рядом с notebooks
# (в Colab: загрузите архив материалов дисциплины и распакуйте его в /content)
import sys, os, numpy as np, matplotlib.pyplot as plt
for p in ("../scripts", "scripts", "/content/scripts", "../detective", "/content/detective"):
    if os.path.isdir(p): sys.path.insert(0, os.path.abspath(p))
from srm_model import SRM, simulate_time, cycle_angle_domain
mot = SRM(); deg = np.deg2rad
print("Модель загружена: ВИД", f"{mot.Ns}/{mot.Nr}", "Udc =", mot.Udc, "В")

## Перебор углов при 1500 об/мин

In [ ]:
on, off = np.meshgrid(deg(np.arange(-20, 41, 2.5)), deg(np.arange(110, 171, 2.5)))
T, Irms, cont, Ipk = cycle_angle_domain(mot, 1500, on, off, i_max=20)
ok = (~cont) & (Ipk <= 21) & (Irms <= 12.5)
Tm = np.where(ok, T, np.nan)
plt.contourf(np.rad2deg(on), np.rad2deg(off), Tm, 20); plt.colorbar(label='момент, Н·м')
plt.xlabel('угол включения, эл. град'); plt.ylabel('угол отключения, эл. град')
j = np.unravel_index(np.nanargmax(Tm), T.shape)
print('лучшее:', np.rad2deg(on[j]), np.rad2deg(off[j]), 'момент', round(float(T[j]), 2), 'Н·м')

## Проверка во временной области и по энергии цикла

In [ ]:
theta_on, theta_off, i_ref = 20.0, 145.0, 20.0     # ЗАДАНИЕ: подставьте свой режим
Tel = 60/1500/mot.Nr
r = simulate_time(mot, 1500, deg(theta_on), deg(theta_off), i_ref, band=0.8, t_end=2*Tel, dt=5e-6)
n = len(r['t'])//2
W = np.trapezoid(r['i'][n:,0]*r['v'][n:,0] - mot.R*r['i'][n:,0]**2, r['t'][n:])
T_W = W*mot.m*mot.Nr/(2*np.pi)
print(f"момент по модели {r['Te'][n:].mean():.2f} Н·м, по энергии цикла {T_W:.2f} Н·м")
print(f"пиковый ток {r['i'][n:,0].max():.2f} А, действующий {np.sqrt(np.mean(r['i'][n:,0]**2)):.2f} А")
plt.plot(r['i'][n:,0], r['psi'][n:,0]); plt.xlabel('Ток, А'); plt.ylabel('ψ, Вб'); plt.grid(alpha=.3)

In [ ]:
import json, pathlib
out = pathlib.Path("results/lab2"); out.mkdir(parents=True, exist_ok=True)
json.dump({"theta_on_deg": theta_on, "theta_off_deg": theta_off, "i_ref": i_ref,
           "band": 0.8, "soft": True, "T_from_W": float(T_W)}, open(out/"lab2_params.json", "w"), indent=1)
print("Сохранено для автопроверки")

**Задание.** Постройте предельную характеристику в двигательном и генераторном режимах (см. `make_figures.fig_limit_curve`) и объясните изменение оптимальных углов со скоростью.